# ANN implentation using tensorflow

Objective: Based on data, we try to predict whether the customer will exit bank or not.

Since this is a binary classification problem, so we are using:

*   Input Layer: independent features
*   Hidden Layer: linear step(z=w*x+b); RELU activation function
*   Output Layer: linear step (x=w*x+b) =>logit; apply sigmoid activation function


Binary cross entropy(Log Loss)
J(w,b)= -1/n summation(yi*log(y^)+(1-yi)*log(1-y^))

Minimize loss using Adam optimizer updating weight and bias





In [ ]:
import tensorflow as tf
print(tf.__version__)

In [ ]:
#import some basic libraries

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
dataset=pd.read_csv('/content/Churn_Modelling.csv')
dataset.head()

In [ ]:
#divide the dataset into dependent and independent features

X=dataset.iloc[:,3:13]
y=dataset.iloc[:,13]

In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
# Feature Engineering
geography=pd.get_dummies(X['Geography'],drop_first=True,dtype='int')
gender=pd.get_dummies(X['Gender'],drop_first=True,dtype='int')

In [ ]:
## concatenate these variables with dataset
X=X.drop(['Geography','Gender'],axis=1)

In [ ]:
X=pd.concat([X,geography,gender],axis=1)
X

In [ ]:
#split dataset into training set and test set

from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test= train_test_split(X,y,test_size=0.2,random_state=0)


In [ ]:
 #feature scaling
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()
X_train=sc.fit_transform(X_train)
X_test=sc.transform(X_test)

In [ ]:
X_train

In [ ]:
X_test

In [ ]:
X_train.shape


keras is high-level API inside tensorflow that simplifies model building,training and evaluating neural network

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import ReLU
from tensorflow.keras.layers import Dropout

In [ ]:
# initialize ANN
classifier = Sequential()

In [ ]:
# Adding input layer
classifier.add(Dense(units=11,activation='relu'))

In [ ]:
# add first hidden layer
classifier.add(Dense(units=7,activation='relu'))

In [ ]:
#add second hiddden layer
classifier.add(Dense(units=6,activation='relu'))

In [ ]:
#add output layer
classifier.add(Dense(units=1,activation='sigmoid'))

In [ ]:
#compile
classifier.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])


In [ ]:
#training the model
model=classifier.fit(X_train,y_train,validation_split=0.33,batch_size=10,epochs=1000)


When I keep epoch=1000, at first accuracy increases and loss decreases but after some point it was constant till 1000,so I used early stopping.

Early stopping will stop the training of model automatically when there is no change in accuracy of model in consecutive iterations.

In [ ]:
#Early stopping
early_stopping=tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    min_delta=0.0001,
    patience=20,
    verbose=1,
    mode='auto',
    baseline=None,
    restore_best_weights=False,
    start_from_epoch=0
)

In [ ]:
model=classifier.fit(X_train,y_train,validation_split=0.33,batch_size=10,epochs=1000,callbacks=early_stopping)


In [ ]:
model.history.keys()

In [ ]:
#summarize history for accuracy
plt.plot(model.history['accuracy'])
plt.plot(model.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train','test'],loc='upper left')
plt.show()

In [ ]:
#summarize history for loss
plt.plot(model.history['loss'])
plt.plot(model.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train','test'],loc='upper left')
plt.show()

In [ ]:
# making predictions and evaluating the performance of the model
y_pred=classifier.predict(X_test)
y_pred=(y_pred>=0.5)

In [ ]:
#make the confusion matrix
from sklearn.metrics import confusion_matrix
cm=confusion_matrix(y_test,y_pred)
cm

In [ ]:
from sklearn.metrics import accuracy_score
score=accuracy_score(y_test,y_pred)

In [ ]:
score

In [ ]:
#get the weights and bias
classifier.get_weights()